# Light Rider Quantum Quickstart

Build a quantum circuit with the [`lightrider`](https://pypi.org/project/lightrider/) Python SDK, check it on a free local simulator, then submit it to an IQM quantum backend through the Light Rider platform.

You need a Light Rider API key (`lr_...`) — create one at [platform.lightriderinc.com/settings/keys](https://platform.lightriderinc.com/settings/keys).

In [ ]:
import sys

major, minor = sys.version_info[:2]
print(f"Python {major}.{minor} detected.")
if major != 3 or minor < 10:
    print("\u26a0\ufe0f This notebook requires Python 3.10 or above. In Colab: Runtime \u2192 Change runtime type.")

In [ ]:
%pip install -q lightrider==1.3.1 requests

import requests

#@markdown Paste your Light Rider API key below, then run this cell.
api_key = "" #@param {type:"string"}
base_url = "https://platform.lightriderinc.com"

api_key = api_key.strip()
if not api_key.startswith("lr_"):
    raise ValueError("Expected a Light Rider API key beginning with 'lr_'. Create one at /settings/keys on the platform.")

session = requests.Session()
session.headers["Authorization"] = f"Bearer {api_key}"

## Build a circuit with the SDK

A Bell pair: Hadamard on qubit 0, then CNOT onto qubit 1, then measure both qubits.

In [ ]:
from lightrider import Circuit

circuit = Circuit(2, name="colab-bell")
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

print(circuit)

## Dry-run locally (free)

The SDK ships local simulators, so you can verify the circuit's physics before spending any credits. A Bell pair should give only `00` and `11`, roughly 50/50.

In [ ]:
from lightrider import get_backend

local = get_backend("statevector")
print("local counts:", local.run(circuit, shots=1000).result().get_counts())

## Pick a platform backend

Valid `backend` values for submission:

- **Mock (free, unlimited):** `iqm-garnet-mock`, `iqm-emerald-mock`, `iqm-sirius-mock`
- **Real QPU (costs credits, requires a prior purchase):** `iqm-garnet`, `iqm-emerald`, `iqm-sirius`

In [ ]:
backend = "rigetti-cepheus-mock"  # change this to the backend you want to use

response = session.post(
    f"{base_url}/api/lr/quantum/submit",
    json={"backend": backend, "circuit": circuit.to_payload(), "shots": 1000},
)
if response.status_code == 402:
    # Billing gate: real QPUs need purchased credits and a sufficient balance.
    detail = response.json()
    raise RuntimeError(f"{detail['error']}: {detail['message']}")
response.raise_for_status()
job = response.json()
print("Job submitted:", job["job_uuid"])
print("Status:", job["status"])

In [ ]:
import time

job_id = job["job_uuid"]

for _ in range(300):
    status_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}")
    status_response.raise_for_status()
    status_data = status_response.json()
    print("Status:", status_data["status"])

    if status_data.get("isInTerminalState"):
        break
    time.sleep(1)
else:
    raise TimeoutError("Job did not complete within 5 minutes. Check platform.lightriderinc.com/jobs for status.")

result_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}/result")
result_response.raise_for_status()
counts = result_response.json()["counts"]
print("Counts:", counts)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(counts.keys(), counts.values())
plt.xlabel("Measurement outcome")
plt.ylabel("Count")
plt.title(f"Bell pair on {backend}")
plt.show()

This circuit creates a **Bell pair** — two qubits entangled so they always agree when measured. You should see only `00` and `11` in the results, never `01` or `10`, roughly split 50/50 — the same distribution the local simulator predicted above. That agreement pattern is the signature of quantum entanglement.

## Billing and plans

Every new signup gets 1000 free Light Rider credits automatically — no payment required. Mock backends (`iqm-garnet-mock`, `iqm-emerald-mock`, `iqm-sirius-mock`) are always free and unlimited regardless of your balance.

Switch `backend` to `iqm-garnet`, `iqm-emerald`, or `iqm-sirius` to run on real quantum hardware — that costs credits, and requires having purchased credits at least once (the free signup credits alone don't unlock real hardware). If you haven't purchased yet, you'll get a 402 (`purchase_required`) instead of a job id. If you have purchased before but your balance has since run out, you'll get a 402 (`insufficient_credits`) instead. Buy credits at [/settings/purchases/quantum-compute](https://platform.lightriderinc.com/settings/purchases/quantum-compute).

Your submitted jobs — from this notebook and from the web dashboard alike — are listed at [platform.lightriderinc.com/jobs](https://platform.lightriderinc.com/jobs).